# Phase 7: Generation Pipeline

Ghép nối Retrieval (Phase 6) với LLM thông qua OpenRouter để tạo pipeline RAG hoàn chỉnh.
Thực hiện sinh câu trả lời cho **tất cả 9 cấu hình** (3 strategies x 3 methods) để chuẩn bị dữ liệu cho Phase 8 đánh giá toàn diện.

In [ ]:
import os, sys, subprocess
from pathlib import Path

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_ROOT = Path('/content/rag-vn-finance')
    if not REPO_ROOT.exists():
        print("Đang tải mã nguồn và cài đặt thư viện lần đầu...")
        subprocess.run(['git', 'clone', 'https://github.com/thong7d/rag-vn-finance.git', str(REPO_ROOT)])
        req_path = REPO_ROOT / 'requirements.txt'
        if req_path.exists():
            os.system(f'pip install -r "{req_path}" -q')
        print("Cài đặt hoàn tất. Đang tự động khởi động lại Kernel...")
        os.kill(os.getpid(), 9)
    else:
        print("Mã nguồn đã tồn tại. Bỏ qua cài đặt...")
else:
    REPO_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())

print(f"Project root: {REPO_ROOT}")
assert REPO_ROOT.exists(), f"Project root not found: {REPO_ROOT}"

src_path = str(REPO_ROOT)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

if IN_COLAB:
    from dotenv import load_dotenv
    load_dotenv('/content/drive/MyDrive/rag-vn-finance/.env')

print("\nColab setup complete.")

## 1. Environment Setup & Config

Load cấu hình và khởi tạo mô hình Embedding.

In [ ]:
import json
import pandas as pd
import faiss
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

from src.utils import load_config, resolve_path, ensure_dir, get_env
from src.indexing import load_bm25_index
from src.retrieval import DenseRetriever, SparseRetriever, HybridRetriever
from src.generation import generate_answer

# ── Config ─────────────────────────────────────────────────────────────────────
config = load_config()

EVAL_SAMPLE = config['evaluation']['eval_sample_size']  # 200 câu
STRATEGIES = config['chunking']['strategies']
METHODS = ["Dense", "Sparse", "Hybrid"]

print(f"Sẽ chạy {len(STRATEGIES) * len(METHODS)} cấu hình trên {EVAL_SAMPLE} câu hỏi.")

# ── Load Embedding Model ────────────────────────────────────────────────────────
model_name = config['embedding']['model_name']
device = config['embedding'].get('device', 'cpu')
if device == 'auto':
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer(model_name, device=device)
print(f"\nLoaded {model_name} on {model.device}")

## 2. Load Evaluation Data

Load tập `qa_pairs_test.parquet` (tập test 20% bất biến từ Phase 5).

In [ ]:
qa_dir = resolve_path(config['synthetic_qa'], 'output_dir')
test_path = os.path.join(qa_dir, 'qa_pairs_test.parquet')
fallback_path = os.path.join(qa_dir, 'qa_pairs_filtered.parquet')

if os.path.exists(test_path):
    df_qa = pd.read_parquet(test_path)
    print(f"Loaded test set: {len(df_qa)} QA pairs")
elif os.path.exists(fallback_path):
    df_qa = pd.read_parquet(fallback_path)
    print(f"Test set not found, using filtered set: {len(df_qa)} QA pairs")
else:
    raise FileNotFoundError("Không tìm thấy QA data. Hãy chạy Phase 5 trước.")

if len(df_qa) > EVAL_SAMPLE:
    df_eval = df_qa.sample(n=EVAL_SAMPLE, random_state=42).reset_index(drop=True)
else:
    df_eval = df_qa.reset_index(drop=True)

print(f"Evaluation set: {len(df_eval)} samples")

## 3. Generation Loop (9 Configs)

Vòng lặp sẽ tự động tải index tương ứng cho từng `strategy`, sau đó chạy lần lượt cả 3 `method` (Dense, Sparse, Hybrid). File `.parquet` sẽ được lưu chung hoặc lưu riêng để dễ quản lý.

In [ ]:
index_base_dir = resolve_path(config['indexing'], 'output_dir')
bm25_base_dir  = resolve_path(config['indexing'], 'bm25_dir')
eval_dir = resolve_path(config['evaluation'], 'output_dir')
ensure_dir(eval_dir)
out_path = os.path.join(eval_dir, 'generation_results_all.parquet')

CHECKPOINT_EVERY = 20
all_results = []

for strategy in STRATEGIES:
    print(f"\n{'='*40}\nĐang xử lý Strategy: {strategy}\n{'='*40}")
    
    # --- 1. Load Indexes cho Strategy --- 
    faiss_path = os.path.join(index_base_dir, strategy, "index.faiss")
    chunk_ids_path = os.path.join(index_base_dir, strategy, "chunk_ids.json")
    metadata_path = os.path.join(index_base_dir, strategy, "metadata.parquet")
    
    if not os.path.exists(faiss_path):
        print(f"Bỏ qua {strategy} vì không tìm thấy FAISS index.")
        continue
        
    faiss_index = faiss.read_index(faiss_path)
    with open(chunk_ids_path, 'r', encoding='utf-8') as f:
        chunk_ids = json.load(f)
        
    df_meta = pd.read_parquet(metadata_path)
    chunk_text_map = dict(zip(df_meta['chunk_id'], df_meta['text']))
    
    dense_retriever = DenseRetriever(faiss_index, chunk_ids, model)
    
    try:
        bm25_index, bm25_chunk_ids = load_bm25_index(bm25_base_dir, strategy)
        sparse_retriever = SparseRetriever(bm25_index, bm25_chunk_ids)
    except FileNotFoundError:
        print(f"Bỏ qua {strategy} vì không tìm thấy BM25 index.")
        continue
        
    hybrid_retriever = HybridRetriever(dense_retriever, sparse_retriever, rrf_k=config['retrieval']['rrf_k'])
    retriever_map = {"Dense": dense_retriever, "Sparse": sparse_retriever, "Hybrid": hybrid_retriever}
    
    # --- 2. Chạy 3 Methods ---
    for method in METHODS:
        print(f"\n---> Method: {method}")
        retriever = retriever_map[method]
        top_k = config['retrieval'][f'top_k_{method.lower()}']
        
        for i, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc=f"{strategy}-{method}"):
            query = row['question']
            gt_answer = row.get('answer', row.get('reference_answer', ''))
            doc_id = row['doc_id']
            
            # 2.1 Retrieve & Lookup
            retrieved = retriever.retrieve(query, top_k=top_k)
            retrieved_ids = [cid for cid, _ in retrieved]
            contexts = [chunk_text_map.get(cid, "") for cid in retrieved_ids if cid in chunk_text_map]
            
            # 2.2 Generate
            try:
                generated_answer = generate_answer(query, contexts)
            except Exception as e:
                print(f"\n[WARNING] Lỗi sinh answer (Row {i}): {e}")
                generated_answer = ""
                
            all_results.append({
                'doc_id': doc_id,
                'question': query,
                'contexts': "\n---\n".join(contexts),
                'generated_answer': generated_answer,
                'reference_answer': gt_answer,
                'strategy': strategy,
                'method': method,
                'retrieved_ids': json.dumps(retrieved_ids, ensure_ascii=False)
            })
            
            # Checkpoint (bên trong vòng lặp)
            if len(all_results) % CHECKPOINT_EVERY == 0:
                pd.DataFrame(all_results).to_parquet(out_path, index=False)
                
        # --- Checkpoint sau khi xong 1 config ---
        pd.DataFrame(all_results).to_parquet(out_path, index=False)
        print(f"  [CHECKPOINT] Đã hoàn thành và lưu kết quả cấu hình {strategy} - {method} (Tổng: {len(all_results)} dòng)")


## 4. Save Final Results

Lưu toàn bộ kết quả cuối cùng (có thể lên tới 1800 dòng) vào file.

In [ ]:
df_results = pd.DataFrame(all_results)
df_results.to_parquet(out_path, index=False)
print(f"\nHOÀN THÀNH! Đã lưu {len(df_results)} kết quả (gồm cả 9 config) vào {out_path}")
display(df_results.head())